# ROGII Wellbore curves recognition: a minimal end-to-end baseline

The task: trace the curves on a scanned well-log raster. For every well (an image of up to ~100 000 rows) the
submission holds the traced curves as polylines; the metric (Curve-PQ) matches them with the annotated curves.

This notebook goes from the raw files to a submission in five short steps:
1. look at the data, 2. cut training crops from batch 1, 3. train a small U-Net that marks curve pixels,
4. predict, turn the pixel map into curves (skeleton -> chains -> linked curves), score a validation batch with
the competition metric and write `submission.csv.zip` for the test wells.

It uses every GPU it finds (the training batch is split between them, the test wells are shared out). It runs top to bottom on a Kaggle GPU notebook in roughly an hour (data prep ~3 min, training ~7 min,
prediction ~30 min) and scores about 0.42 on the public leaderboard. Everything is deliberately plain: the
idea is to have something you can read in one sitting and then improve.

In [1]:
import json, os, sys, time, zipfile, zlib
from multiprocessing import Pool

import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image

Image.MAX_IMAGE_PIXELS = None                       # the images are huge
SEED = 0
# settings (the defaults reproduce the baseline; smaller values give a quick smoke test)
TRAIN_WELLS, CROPS, POSITIVE = int(os.environ.get("TRAIN_WELLS", 500)), 64, 46   # wells of batch 1, crops per well, how many centred on a curve
EPOCHS = int(os.environ.get("EPOCHS", 14))
VAL_WELLS = int(os.environ.get("VAL_WELLS", 60))        # wells of batch 12 used for a local score
TEST_LIMIT = int(os.environ.get("TEST_LIMIT", 0))       # 0 = all test wells
PROCS = min(4, os.cpu_count() or 1)

def find(name):
    """Path of a file or folder of the competition input (Kaggle mounts it under /kaggle/input/...)."""
    for root in (os.environ.get("ROGII_INPUT", "/kaggle/input"),):
        for dirpath, dirnames, filenames in os.walk(root):
            if name in dirnames or name in filenames:
                return os.path.join(dirpath, name)
            if dirpath[len(root):].count(os.sep) > 2:
                dirnames[:] = []
    raise FileNotFoundError(name)

## 1. The data
* `train_images_bNN/<id>.tif` - the scan (RGBA), `train_masks_bNN/<id>.png` - 1 where an annotated curve passes
  (the masks only mark curves the annotators traced: many real curves are not annotated),
* `train_curves_bNN.csv` - `id, curves, zone`: the curves as polylines `[[x, y], ...]` (a `null` inside a curve is a gap)
  and the evaluation zone (rectangles of the tracks the annotators worked in; only the zone is scored),
* `test_images_01/`, `test_images_02/`, `sample_submission.csv` (`id, curves`) and `metric.py`, the competition metric.

In [2]:
INPUT = os.path.dirname(find("sample_submission.csv"))
print("input:", INPUT)
w = sorted(os.listdir(find("train_images_b01")))[0][:-4]
im = Image.open(f"{find('train_images_b01')}/{w}.tif"); mk = np.asarray(Image.open(f"{find('train_masks_b01')}/{w}.png"))
print(f"well {w}: image {im.size[0]} x {im.size[1]} px ({im.mode}), mask share of curve pixels {mk.mean():.3f}")
row = pd.read_csv(find("train_curves_b01.csv"), nrows=1).iloc[0]
print("curves:", len(json.loads(row["curves"])), "| first points of curve 0:", json.loads(row["curves"])[0][:3], "| zone:", row["zone"])

input: /kaggle/input/rogii-wellbore-test
well 0018ca82: image 1125 x 25596 px (RGBA), mask share of curve pixels 0.006
curves: 1 | first points of curve 0: [[18.0, 14.0], [19.0, 15.0], [37.0, 16.0]] | zone: [[-22.6,0.4,377.2,25594.9]]


## 2. Workers: from a pixel map to curves (CPU)
The network gives a probability for every pixel (we run it on 1024-pixel tiles with a small overlap).
We threshold it, thin the mask to a one-pixel **skeleton**, cut the skeleton into **chains** between junctions
and simplify them (Douglas-Peucker, 1 px). Pieces of one connected skeleton component are one curve. Pieces that
continue each other further down the well (similar x, similar colour) are **linked** into a single curve with a
`null` gap between them. The submission may use `null`: the points around it are not joined.
Linking is where most of the score comes from; breaking one curve into pieces or merging several curves into one
is what costs points.

In [3]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from skimage.measure import approximate_polygon
from skimage.morphology import skeletonize

TILE, OVERLAP, THRESHOLD = 1024, 32, 0.15
MIN_LINK, MIN_TRACK, MAX_CURVES = 20, 100, 300       # the metric accepts at most 300 curves per well
GAP_MAX, COL_MAX = 1500, 110                          # linking: largest gap (rows), largest colour distance

def curves_from_mask(mask, image):
    H, W = mask.shape; ys, xs = np.nonzero(skeletonize(mask))
    if len(ys) < MIN_LINK: return []
    key = ys.astype(np.int64) * W + xs; n = len(key); edges = []
    for dy, dx in ((0, 1), (1, -1), (1, 0), (1, 1)):                  # 8-neighbourhood of every skeleton pixel
        ok = (xs + dx >= 0) & (xs + dx < W) & (ys + dy < H); nk = key[ok] + dy * W + dx
        pos = np.searchsorted(key, nk); pos[pos >= n] = n - 1; hit = key[pos] == nk
        edges.append((np.flatnonzero(ok)[hit], pos[hit]))
    a = np.concatenate([e[0] for e in edges]); b = np.concatenate([e[1] for e in edges])
    deg = np.bincount(a, minlength=n) + np.bincount(b, minlength=n)
    ncomp, comp = connected_components(coo_matrix((np.ones(len(a), np.int8), (a, b)), shape=(n, n)), directed=False)
    nj = (deg[a] < 3) & (deg[b] < 3)                                   # edges that do not touch a junction -> chains
    _, chain = connected_components(coo_matrix((np.ones(nj.sum(), np.int8), (a[nj], b[nj])), shape=(n, n)), directed=False)
    order = np.argsort(chain, kind="stable"); groups = np.split(order, np.flatnonzero(np.diff(chain[order])) + 1)
    size = np.bincount(comp); parts = {}
    for g in groups:
        if len(g) < 3 or deg[g[0]] >= 3 or size[comp[g[0]]] < MIN_LINK: continue
        pts = approximate_polygon(np.column_stack([xs[g], ys[g]]).astype(float), 1.0)
        if len(pts) >= 2: parts.setdefault(int(comp[g[0]]), []).append(pts)
    comps = np.array(sorted(parts))
    if len(comps) == 0: return []
    root = {int(c): int(c) for c in comps}
    def find_root(x):
        while root[x] != x: root[x] = root[root[x]]; x = root[x]
        return x
    if len(comps) > 1:                                                 # link: the end of a piece -> the start of another one below
        o = np.argsort(comp, kind="stable"); sc = comp[o]; st = np.flatnonzero(np.r_[True, sc[1:] != sc[:-1]]); en = np.r_[st[1:], n] - 1
        first = dict(zip(sc[st].tolist(), o[st].tolist())); last = dict(zip(sc[st].tolist(), o[en].tolist()))
        fi = np.array([first[int(c)] for c in comps]); la = np.array([last[int(c)] for c in comps])
        ty, tx, by, bx = ys[fi].astype(float), xs[fi].astype(float), ys[la].astype(float), xs[la].astype(float)
        colour = np.stack([np.bincount(comp, weights=image[ys, xs, k].astype(float), minlength=ncomp) / np.maximum(size, 1) for k in range(3)], 1)[comps]
        by_top = np.argsort(ty); tys = ty[by_top]; cand = []
        for i in range(len(comps)):
            js = by_top[np.searchsorted(tys, by[i] - 15):np.searchsorted(tys, by[i] + GAP_MAX, side="right")]; js = js[js != i]
            if len(js) == 0: continue
            gap = np.maximum(ty[js] - by[i], 0); dx = np.abs(tx[js] - bx[i]); cd = np.linalg.norm(colour[js] - colour[i], axis=1)
            m = (dx <= 12 + 0.06 * gap) & (cd <= COL_MAX)
            cand += [(d + 0.05 * g_ + 0.3 * c, i, int(j)) for j, g_, d, c in zip(js[m], gap[m], dx[m], cd[m])]
        cand.sort(); has_next, has_prev = set(), set()
        for _, i, j in cand:                                           # cheapest links first, one successor and one predecessor each
            if i in has_next or j in has_prev: continue
            ri, rj = find_root(int(comps[i])), find_root(int(comps[j]))
            if ri == rj: continue
            root[ri] = rj; has_next.add(i); has_prev.add(j)
    tracks = {}
    for c in comps: tracks.setdefault(find_root(int(c)), []).append(int(c))
    items = sorted([(sum(int(size[c]) for c in v), v) for v in tracks.values() if sum(int(size[c]) for c in v) >= MIN_TRACK], key=lambda t: -t[0])
    out = []
    for _, v in items[:MAX_CURVES]:
        cur, first_part = [], True
        for c in v:
            for p in parts[c]:
                if not first_part: cur.append(None)
                first_part = False; cur += np.rint(p).astype(int).tolist()   # whole pixels keep the file small
        out.append(cur)
    return out

# the crop sampler of step 3 also lives here, so that the workers know it
C = 256
def crops_of(wid):
    rng = np.random.default_rng(zlib.crc32(wid.encode()))
    X = np.asarray(Image.open(f"{find('train_images_b01')}/{wid}.tif").convert("RGB"))
    M = np.asarray(Image.open(f"{find('train_masks_b01')}/{wid}.png")) > 0
    H, W = M.shape
    xs_, ys_ = np.zeros((CROPS, C, C, 3), np.uint8) + 255, np.zeros((CROPS, C, C), np.uint8)
    yy, xx = np.nonzero(M)
    if len(yy) == 0:
        return xs_, ys_
    lo, hi = int(yy.min()), int(yy.max())
    for t in range(CROPS):
        if t < POSITIVE:
            j = rng.integers(len(yy)); y0 = int(yy[j]) - int(rng.integers(32, C - 32)); x0 = int(xx[j]) - int(rng.integers(32, C - 32))
        else:
            y0 = int(rng.integers(lo, max(lo + 1, hi - C))); x0 = int(rng.integers(0, max(1, W - C + 1)))
        y0 = max(0, min(y0, H - C)) if H >= C else 0; x0 = max(0, min(x0, W - C)) if W >= C else 0
        h, w_ = min(C, H - y0), min(C, W - x0)
        xs_[t, :h, :w_] = X[y0:y0 + h, x0:x0 + w_]; ys_[t, :h, :w_] = M[y0:y0 + h, x0:x0 + w_]
    return xs_, ys_

def post(args):
    """CPU part, run in a worker: probability map saved by the main process -> JSON curves."""
    path, prob_path = args
    image = np.asarray(Image.open(path).convert("RGB")); prob = np.load(prob_path); os.remove(prob_path)
    return json.dumps(curves_from_mask(prob >= int(THRESHOLD * 255), image), separators=(",", ":"))

# The workers must be forked BEFORE the main process touches the GPU (a CUDA context does not survive a fork).
# The GPUs stay in the main process; the workers only do the CPU work above.
POOL = Pool(PROCS)

## 3. Training crops
A whole well is too large for a network, so we cut 256 x 256 crops: most of them centred on a random curve pixel
(with a random shift), the rest anywhere inside the annotated depth range. The sampler is `crops_of` above.

In [4]:
ids = sorted(f[:-4] for f in os.listdir(find("train_images_b01")))
ids = list(np.random.default_rng(SEED).permutation(ids)[:TRAIN_WELLS])
t0 = time.time()
res = POOL.map(crops_of, ids, chunksize=4)
Xtr = np.concatenate([r[0] for r in res]); Ytr = np.concatenate([r[1] for r in res]); del res
print(f"{len(Xtr)} crops from {len(ids)} wells in {time.time() - t0:.0f} s | curve pixels: {Ytr.mean():.3f}")

32000 crops from 500 wells in 605 s | curve pixels: 0.038


## 4. A small U-Net
1.56 M parameters. The loss is binary cross-entropy (curve pixels are rare, so they get weight 4) plus Dice.

In [ ]:
def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))

class UNet(nn.Module):
    def __init__(s, ch=(16, 32, 64, 128, 192)):
        super().__init__()
        s.enc = nn.ModuleList([block(3, ch[0])] + [block(ch[k], ch[k + 1]) for k in range(len(ch) - 1)])
        s.dec = nn.ModuleList([block(ch[k + 1] + ch[k], ch[k]) for k in range(len(ch) - 1)])
        s.out = nn.Conv2d(ch[0], 1, 1)
    def forward(s, x):
        x = (x - 0.5) / 0.5; f = []
        for k, e in enumerate(s.enc):
            x = e(x if k == 0 else F.max_pool2d(x, 2)); f.append(x)
        x = f[-1]
        for k in range(len(s.dec) - 1, -1, -1):
            x = F.interpolate(x, size=f[k].shape[-2:], mode="bilinear", align_corners=False)
            x = s.dec[k](torch.cat([x, f[k]], 1))
        return s.out(x)

dev = "cuda:0" if torch.cuda.is_available() else "cpu"
GPUS = [int(g) for g in os.environ["ROGII_GPUS"].split(",")] if os.environ.get("ROGII_GPUS") else list(range(torch.cuda.device_count()))   # all GPUs (Kaggle: 2 x T4)
amp = torch.bfloat16 if (dev != "cpu" and torch.cuda.is_bf16_supported()) else torch.float16
torch.manual_seed(SEED); torch.backends.cudnn.benchmark = True
net = UNet().to(dev)
class Amp(nn.Module):
    """DataParallel runs every replica in its own thread, and autocast does not follow into those threads: switch it on inside."""
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, x):
        with torch.autocast("cuda", dtype=amp, enabled=(dev != "cpu")): return s.m(x)
train_gpus = sorted(set(GPUS))
model = nn.DataParallel(Amp(net), device_ids=train_gpus) if len(train_gpus) > 1 else Amp(net)      # the batch of 64 is split between the GPUs
if len(train_gpus) > 1:                                                                           # a safety net: fall back to one GPU instead of failing
    try:
        model(torch.zeros(2 * len(train_gpus), 3, 64, 64, device=dev))
    except Exception as e:
        print("DataParallel did not work, training on one GPU:", repr(e)[:150]); model = Amp(net)
print("GPUs:", GPUS if GPUS else "none (CPU)")
BS = 64; steps = len(Xtr) // BS
opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=max(1, EPOCHS * steps), pct_start=0.15)
scaler = torch.amp.GradScaler(enabled=(dev != "cpu" and amp == torch.float16))
pos_weight = torch.tensor(4.0, device=dev)

def loss_fn(logits, y):
    logits = logits.float(); p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3)); dice = 1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)
    return F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight) + dice.mean()

if os.environ.get("WEIGHTS"):                                    # skip training: use given weights (a quick check)
    net.load_state_dict(torch.load(os.environ["WEIGHTS"], map_location=dev))
else:
    rng = np.random.default_rng(SEED); t0 = time.time()
    for ep in range(EPOCHS):
        model.train(); perm = rng.permutation(len(Xtr)); total = 0.0
        for b in range(steps):
            idx = np.sort(perm[b * BS:(b + 1) * BS])
            x = torch.from_numpy(Xtr[idx]).to(dev).permute(0, 3, 1, 2).float() / 255
            y = torch.from_numpy(Ytr[idx]).to(dev).float().unsqueeze(1)
            if rng.random() < .5: x, y = x.flip(3), y.flip(3)                       # augmentation: flips and brightness
            if rng.random() < .5: x, y = x.flip(2), y.flip(2)
            x = (x * (0.85 + 0.3 * torch.rand(len(x), 1, 1, 1, device=dev))).clamp_(0, 1)
            logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            total += loss.item() if b % 50 == 0 else 0
        print(f"epoch {ep + 1}/{EPOCHS} loss~{total / max(1, steps // 50):.3f}  {time.time() - t0:.0f} s", flush=True)
    torch.save(net.state_dict(), "unet.pt")
import copy
net.eval(); del Xtr, Ytr                                         # inference mode: BatchNorm must use its running statistics
devices = [f"cuda:{g}" for g in GPUS] or ["cpu"]
nets = [copy.deepcopy(net).to(d).eval() for d in devices]       # one copy of the network per GPU

GPUs: [0, 1]
epoch 1/14 loss~1.494  864 s
epoch 2/14 loss~0.808  1734 s
epoch 3/14 loss~0.527  2604 s
epoch 4/14 loss~0.522  3474 s
epoch 5/14 loss~0.481  4344 s
epoch 6/14 loss~0.505  5214 s
epoch 7/14 loss~0.507  6083 s
epoch 8/14 loss~0.467  6953 s
epoch 9/14 loss~0.468  7805 s
epoch 10/14 loss~0.455  8640 s
epoch 11/14 loss~0.451  9476 s
epoch 12/14 loss~0.421  10311 s
epoch 13/14 loss~0.459  11146 s
epoch 14/14 loss~0.440  11981 s


### Prediction on tiles
The network gives a probability for every pixel; we run it on 1024-pixel tiles with a small overlap.

In [6]:
@torch.no_grad()
def predict(image, k=0):
    """Curve probability 0..255 for every pixel of an (H, W, 3) image, computed on GPU number k."""
    d, m = devices[k], nets[k]
    H, W, _ = image.shape; out = np.zeros((H, W), np.uint8); side = TILE + 2 * OVERLAP
    tiles = [(y, x) for y in range(0, H, TILE) for x in range(0, W, TILE)]
    for b in range(0, len(tiles), 4):
        sel = tiles[b:b + 4]; batch = np.full((len(sel), side, side, 3), 255, np.uint8); origin = []
        for j, (y, x) in enumerate(sel):
            y0, x0 = max(0, y - OVERLAP), max(0, x - OVERLAP)
            crop = image[y0:min(H, y + TILE + OVERLAP), x0:min(W, x + TILE + OVERLAP)]
            batch[j, :crop.shape[0], :crop.shape[1]] = crop; origin.append((y - y0, x - x0))
        t = torch.from_numpy(batch).to(d).permute(0, 3, 1, 2).float() / 255
        with torch.autocast("cuda", dtype=amp, enabled=(d != "cpu")):
            p = (torch.sigmoid(m(t).float()) * 255).round().byte().cpu().numpy()[:, 0]
        for j, (y, x) in enumerate(sel):
            hh, ww = min(TILE, H - y), min(TILE, W - x); oy, ox = origin[j]
            out[y:y + hh, x:x + ww] = p[j, oy:oy + hh, ox:ox + ww]
    return out

from concurrent.futures import ThreadPoolExecutor
GPU_THREADS = [ThreadPoolExecutor(1) for _ in devices]          # one thread per GPU: the GPUs work in parallel

def gpu_job(path, k):
    image = np.asarray(Image.open(path).convert("RGB")); prob_path = f"/tmp/prob/{os.path.basename(path)[:-4]}.npy"
    np.save(prob_path, predict(image, k)); return prob_path

def trace_many(paths):
    """JSON curves for a list of image paths. The networks run here, one thread per GPU; skeletons and links run in the worker pool.
    The wells go in small chunks: while the GPUs work on a chunk, the workers finish the previous one."""
    os.makedirs("/tmp/prob", exist_ok=True); out, waiting, C = {}, [], len(devices) * PROCS * 2
    for c0 in range(0, len(paths), C):
        chunk = paths[c0:c0 + C]
        futs = [GPU_THREADS[i % len(devices)].submit(gpu_job, p, i % len(devices)) for i, p in enumerate(chunk)]
        for p, r in waiting: out[p] = r.get()                    # the CPU part of the previous chunk
        waiting = [(p, POOL.apply_async(post, ((p, f.result()),))) for p, f in zip(chunk, futs)]
        print(f"  {c0 + len(chunk)} / {len(paths)}", flush=True)
    for p, r in waiting: out[p] = r.get()
    return [out[p] for p in paths]

### A local score on a validation batch
`train_curves_b12.csv` has the same columns as the hidden solution (`id, curves, zone`), so batch 12 is a natural
validation set. It was not used for training. The competition metric is `metric.py` from the competition files.

In [7]:
try:
    sys.path.insert(0, os.path.dirname(find("metric.py")))
    import metric
except FileNotFoundError:
    metric = None
    print("metric.py is not in the competition files: the local score is skipped")
if metric is not None:
    val = pd.read_csv(find("train_curves_b12.csv"), dtype={"id": str})
    val = val.sample(min(VAL_WELLS, len(val)), random_state=SEED).reset_index(drop=True); val["Usage"] = "Public"
    t0 = time.time()
    sub_val = pd.DataFrame({"id": val["id"], "curves": trace_many([f"{find('train_images_b12')}/{i}.tif" for i in val["id"]])})
    print(f"{len(val)} validation wells traced in {time.time() - t0:.0f} s")
    print("validation score (Curve-PQ, windows of 1600 rows): %.4f" % metric.score(val, sub_val, "id"))

  16 / 60
  32 / 60
  48 / 60
  60 / 60
60 validation wells traced in 762 s
validation score (Curve-PQ, windows of 1600 rows): 0.4323


## 5. Predict the test wells and write the submission
The submission is a csv with `id, curves` (curves = a JSON list of polylines). A zip of the csv is accepted and is
about 3.5 times smaller. Use whole numbers for the coordinates (`[12,34]`, not `[12.0,34.0]`): the files are big.

In [8]:
paths = sorted(f"{d}/{f}" for d in (find("test_images_01"), find("test_images_02")) for f in os.listdir(d) if f.endswith(".tif"))
if TEST_LIMIT: paths = paths[:TEST_LIMIT]
t0 = time.time(); rows = list(zip([os.path.basename(p)[:-4] for p in paths], trace_many(paths)))
print(f"{len(paths)} test wells traced in {time.time() - t0:.0f} s")
sample = pd.read_csv(find("sample_submission.csv"), dtype={"id": str})
got = dict(rows); out = pd.DataFrame({"id": sample["id"], "curves": [got.get(i, "[]") for i in sample["id"]]})
with zipfile.ZipFile("submission.csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("submission.csv", out.to_csv(index=False))
print("submission.csv.zip:", f"{os.path.getsize('submission.csv.zip') / 1e6:.0f} MB,", len(out), "wells")

  16 / 1000
  32 / 1000
  48 / 1000
  64 / 1000
  80 / 1000
  96 / 1000
  112 / 1000
  128 / 1000
  144 / 1000
  160 / 1000
  176 / 1000
  192 / 1000
  208 / 1000
  224 / 1000
  240 / 1000
  256 / 1000
  272 / 1000
  288 / 1000
  304 / 1000
  320 / 1000
  336 / 1000
  352 / 1000
  368 / 1000
  384 / 1000
  400 / 1000
  416 / 1000
  432 / 1000
  448 / 1000
  464 / 1000
  480 / 1000
  496 / 1000
  512 / 1000
  528 / 1000
  544 / 1000
  560 / 1000
  576 / 1000
  592 / 1000
  608 / 1000
  624 / 1000
  640 / 1000
  656 / 1000
  672 / 1000
  688 / 1000
  704 / 1000
  720 / 1000
  736 / 1000
  752 / 1000
  768 / 1000
  784 / 1000
  800 / 1000
  816 / 1000
  832 / 1000
  848 / 1000
  864 / 1000
  880 / 1000
  896 / 1000
  912 / 1000
  928 / 1000
  944 / 1000
  960 / 1000
  976 / 1000
  992 / 1000
  1000 / 1000
1000 test wells traced in 9810 s
submission.csv.zip: 251 MB, 1000 wells
